In [ ]:
from data_gatherer.data_gatherer import DataGatherer
from data_gatherer.llm.response_schema import *
import pandas as pd

In [ ]:
df = pd.read_csv('tables/datasets-Mar_3rd_2026.tsv', sep='\t')
df.columns

In [ ]:
# make sure that the alternative urls col always contains access url, otherwise print False
for index, row in df.iterrows():
    access_url = row['Access URL']
    alternative_urls = row['Alternative URLs']
    if pd.notna(alternative_urls):
        assert isinstance(alternative_urls, str), f"Alternative URLs should be a str, but got {type(alternative_urls)}"
        assert access_url in alternative_urls, f"Row: {index}. Access URL {access_url} is not in Alternative URLs {alternative_urls}"

# create a new df with one row for each alternative url
expanded_rows = []
for index, row in df.iterrows():
    alternative_urls = row['Alternative URLs']
    if pd.isna(alternative_urls):
        expanded_rows.append(row)
    else:
        alt_urls_list = eval(alternative_urls) if isinstance(alternative_urls, str) else []
        #print(alt_urls_list)
        for alt_url in alt_urls_list:
            new_row = row.copy()
            new_row['dataset_webpage'] = alt_url
            expanded_rows.append(new_row)

new_df = pd.DataFrame(expanded_rows)
print(f"New df with {len(new_df)} rows")
new_df.head()

In [ ]:
new_df['download_link'] = None

In [ ]:
# dg = DataGatherer(llm_name="claude-haiku-4-5-20251001", log_level='DEBUG', save_to_cache=True, load_from_cache=True)
dg = DataGatherer(llm_name="claude-haiku-4-5", log_level='INFO', save_to_cache=True, load_from_cache=False)

In [ ]:
outputs = dg.process_metadata(
    new_df[:2],
    pass_cols_to_prompt=['Study Name', 'Abbreviation', 'Diseases Included', 'Sample Size',
       'FAIR Compliance Notes', 'Notes', 'Resource Type', 'Coarse Data Modality', 'Granular Data Modality'],
    display_type='ipynb',
    interactive=True, 
    return_metadata=True,
    use_portkey=False,
    prompt_name='Claude_StudyPage_SanityCheck_rationale',
    response_format=study_sanity_check_w_rationale_schema_claude,
    force_js_load=True,
    profile_dir='~/.firefox-addi-profiler',
    timeout=20,
    add_sitemap_to_prompt=True,
    from_metadata_to_publication_corpus=True
)

In [ ]:
outputs